In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip "/content/drive/MyDrive/Derma_detection/Detectron 2/3classes_data_detectron2.zip"

###**Detectron2 Upload**


In [ ]:
!pip install pyyaml==5.1
!pip install torch==1.8.0+cu101 torchvision==0.9.0+cu101 -f https://download.pytorch.org/whl/torch_stable.html
#install old version of pytorch since detectron2 hasn't released packages for pytorch 1.9

In [ ]:
!pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu101/torch1.8/index.html
# After this step it will ask you to restart the runtime, please do it.

In [ ]:
import torch
assert torch.__version__.startswith("1.8")
import torchvision
import cv2
import os
import numpy as np
import json
import random
import matplotlib.pyplot as plt
%matplotlib inline

from detectron2.structures import BoxMode
from detectron2.data import DatasetCatalog, MetadataCatalog
print(torch.__version__)
print(torchvision.__version__)

### Burada, veri seti kendi özel formatındadır, bu nedenle onu ayrıştırmak ve onu dedektör2'nin standart formatına hazırlamak için bir fonksiyon yazıyoruz. Kullanıcı, özel biçimde bir veri kümesi kullanırken böyle bir işlevi kullanmalıdır.

###**Data Optimization**

Özel formata göre ayarlanacak

In [ ]:
from glob import glob
import imageio


def get_data_dicts(directory, classes):
    dataset_dicts = []
    for filename in [file for file in os.listdir(directory) if file.endswith('.json')]:
        json_file = os.path.join(directory, filename)
        with open(json_file) as f:
            img_anns = json.load(f)

        record = {}

        imagename = glob(diretory + "/*jpg")
        img = imageio.imread(imagename)
        size1 = img.shape[0]
        size2 = img.shape[1]


        filename = os.path.join(directory, img_anns["image_path"]) #imagePath faster RCNN\images\
        record["file_name"] = filename
        record["height"] = size1
        record["width"] = size2
        annos = img_anns["shapes"] #shapes
        objs = []
        for anno in annos:
            px = [a[0] for a in anno['coords']] # x coord
            py = [a[1] for a in anno['coords']] # y-coord
#            poly = [(x, y) for x, y in zip(px, py)] # poly for segmentation
#            poly = [p for x in poly for p in x]

            obj = {
                "bbox": [np.min(px), np.min(py), np.max(px), np.max(py)],
                "bbox_mode": BoxMode.XYXY_ABS,
                "category_id": classes.index(anno['label']),
                "iscrowd": 0
            }
            objs.append(obj)
        record["annotations"] = objs
        dataset_dicts.append(record)

    return dataset_dicts

Burada classlar eklenecek


In [ ]:
classes = ['canis','rubrum','tonsurans']

data_path = '/content/content/Faster rcnn data dermo/'

for d in ["train"]:
    DatasetCatalog.register(
        "my_" + d,
        lambda d=d: get_data_dicts(data_path+d, classes)
    )
    MetadataCatalog.get("my_" + d).set(thing_classes=classes)

microcontroller_metadata = MetadataCatalog.get("my_train")


In [ ]:
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import ColorMode, Visualizer
from matplotlib.patches import Rectangle
from  detectron2.evaluation import COCOEvaluator

###**Yapılandırma Dosyasını Oluşturma**

Class sayısı ayarlanacak


In [ ]:
cfg = get_cfg() # Yapılandırma Dosyasını Oluşturuyoruz
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml")) # Yapılandırma Dosyasını Çeker ve Yapılandırma Dosyasına Ekler
cfg.DATASETS.TRAIN = ("my_train",) # Train Verilerimiz Yapılandırma Dosyasına Kaydeder
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 2 # Çalışan Sayısı
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml") # Ağırlıkları Çeker ve Yapılandırma Dosyasına Ekler
cfg.SOLVER.IMS_PER_BATCH = 8 # Batch Size
cfg.SOLVER.BASE_LR = 0.001 # Learning Rate (Öğrenme Oranı)
cfg.SOLVER.GAMMA = 0.05 # Learning Rate Azaltma Çarpımı
cfg.SOLVER.STEPS = [500] # Learning Rate Azaltma Adım Sayısı
cfg.TEST.EVAL_PERIOD = 200 # Eğitim Sırasında Modeli Değerlendirmek İçin Adım Sayısı

cfg.SOLVER.MAX_ITER = 5000 # İterasyon Sayısı
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 3 # Sınıf Sayısı

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True) # Model Sonucu İçin Klasör Oluşturur
trainer = DefaultTrainer(cfg) # Modeli Train Moduna Geçirir Yapılandırma Dosyası ile Birlikte
trainer.resume_or_load(resume=False) # Model Eğitimine 0'dan Başlamak İçin False Yapıyoruz

In [ ]:
trainer.train() # Model Eğitimine Başlar

In [ ]:
from distutils.dir_util import copy_tree
copy_tree("/content/output", "/content/drive/MyDrive/Detectron 2/10000 iteration output")

In [ ]:
import detectron2
detectron2.evaluation.CityscapesInstanceEvaluator("/content/output/last_checkpoint")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir output

###**Eğitilmiş Model testi**

In [ ]:
#cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth") # Ağırlık Dosyamızı Yapılandırma Dosyamıza Kaydeder
cfg.MODEL.WEIGHTS = "/content/drive/MyDrive/Derma_detection/Detectron 2/deneme train/model_final.pth"    #kayıtlı modeli çalıştırma
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.6 # Test için Eşik Değerimiz
cfg.DATASETS.TEST = ("my_test") # Tets Verilerimiz Yapılandırma Dosyasına Kaydeder
predictor = DefaultPredictor(cfg) #Modeli Test Moduna Geçirir Yapılandırma Dosyası ile Birlikte
test_dataset_dicts = get_data_dicts(data_path+'test', classes)
evaluator =detectron2.evaluation.COCOEvaluator("my_test",cfg,output_dir=cfg.OUTPUT_DIR)
test_metadata = MetadataCatalog.get("my_test")
val_loader = build_detection_test_loader(cfg,"my_test")
result = inference_on_dataset(trainer.model, val_loader, evaluator)


In [ ]:
test_dataset_dicts

In [ ]:
evaluator = detectron2.evaluation.COCOEvaluator("my_train", cfg, False, output_dir=cfg.OUTPUT_DIR)
cfg.MODEL.WEIGHTS="/content/output/model_final.pth"
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.2   # set a custom testing threshold
#cfg.INPUT.MASK_FORMAT='polygon'
val_loader = build_detection_test_loader(cfg, Data_Resister_valid)
inference_on_dataset(trainer.model, val_loader, evaluator)

In [ ]:
for d in ["test"]:
    DatasetCatalog.register(
        "my_" + d,
        lambda d=d: get_data_dicts(data_path+d, classes)
    )
    MetadataCatalog.get("my_" + d).set(thing_classes=classes)

microcontroller_metadata = MetadataCatalog.get("test")

cfg.DATASETS.TEST = ("my_test")

In [ ]:
from  detectron2.evaluation import COCOEvaluator
cfg.DATASETS.TEST = ("my_test")
evaluator = COCOEvaluator("my_test", cfg, False, output_dir="./output/")
val_loader = build_detection_test_loader(cfg, val_dataset)
inference_on_dataset(trainer.model, val_loader, evaluator)

In [ ]:
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.6 # Test için Eşik Değerimiz

In [ ]:
os.mkdir("predict annot")

In [ ]:

import random
import imageio
from matplotlib.pyplot import figure
namelist = {}
for d in  test_dataset_dicts:     #random.sample(test_dataset_dicts,1)

    img = imageio.imread(d["file_name"])

    outputs = predictor(img)
    v = Visualizer(img[:, :, ::-1],
                   metadata=microcontroller_metadata,
                   scale=0.8,
                  # instance_mode=ColorMode.IMAGE_BW # removes the colors of unsegmented pixels
    )
    v = v.draw_instance_predictions(outputs["instances"].to("cpu"))

    #plt.figure(figsize = (10, 10))
    #plt.axis("off")
    #plt.imshow(v.get_image()[:, :, ::-1])
    #plt.title(d["file_name"].split("/")[5])       #file name ayırma
    #plt.show()


#               tahmin değerlerini txt dosyasına  yazdırma
    b = str(outputs["instances"])
    predictannot = b.split("tensor(")[1].strip(" device='cuda:0'), scores:")
    print(b,"\n")# dosya formatını görmek isterseniz açın
#    print(predictannot)
    filename = str(d["file_name"].split("/")[5].strip(".jpg"))
    #print(str(d["file_name"].split("/")[5]))
#    print(filename)

    dosya = open("/content/predict annot/"+filename +".json", "w")
    if predictannot == "[], device='cuda:0', size=(0, 4":
       predictannot = "(0,0)"
    dosya.write(predictannot)
    dosya.close()


In [ ]:
from distutils.dir_util import copy_tree
copy_tree("/content/predict annot", "/content/drive/MyDrive/Derma_detection/Detectron 2/deneme train/predictannot")

In [ ]:
def test(instance, cfg, trainer, test_instance):
  cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
  cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7   # TODO What is this
  cfg.DATASETS.TEST = (test_instance, )
  predictor = DefaultPredictor(cfg)
  evaluator = COCOEvaluator(test_instance, cfg, False, output_dir=cfg.OUTPUT_DIR)
  val_loader = build_detection_test_loader(cfg, test_instance)
  result = inference_on_dataset(trainer.model, val_loader, evaluator)

In [ ]:
os.mkdir("predict annot3")

In [ ]:
from matplotlib.pyplot import figure
namelist = {}
for d in  random.sample(test_dataset_dicts,1):     #random.sample(test_dataset_dicts,1)

    img = cv2.imread(d["file_name"])

    outputs = predictor(img)
    v = Visualizer(img[:, :, ::-1],
                   metadata=microcontroller_metadata,
                   scale=0.8,
                  # instance_mode=ColorMode.IMAGE_BW # removes the colors of unsegmented pixels
    )
    v = v.draw_instance_predictions(outputs["instances"].to("cpu"))

    plt.figure(figsize = (10, 10))
    plt.axis("off")
    plt.imshow(cv2.cvtColor(v.get_image()[:, :, ::-1], cv2.COLOR_BGR2RGB))
    plt.title(d["file_name"].split("/")[4])       #file name ayırma
    plt.show()


##   for x in range(len(outputs["instances"])):
#      #print(x)
#      deneme = outputs["instances"][x]
#      print(deneme)


#               orjinal konumları yazdırma
#    print(d)
#    annot = str(d)
#    print(annot,"\n")
#    print(annot.split("annotations")[1])





#               tahmin değerlerini txt dosyasına  yazdırma
    b = str(outputs["instances"])
    predictannot = b.split("tensor")[1].strip(" device='cuda:0')), scores:")
    print(b,"\n")# dosya formatını görmek isterseniz açın
    print(predictannot)
    filename = str(d["file_name"].split("/")[4].strip(".jpg"))
    print(filename)

    dosya = open("/content/predict annot3/"+filename +".json", "w")
    dosya.write(predictannot)
    dosya.close()


In [ ]:
"""import pandas as od
import numpy as np
img = cv2.imread("/content/faster RCNN/test/airplane_217.jpg")
img = cv2.resize(img, (256,256))
df = pd.read_csv("/content/predict annot/airplane_217.txt")

#plt.imshow(img)
for row in df.iterrows():
    x1 = int(row[1][0].split(" ")[0])
    y1 = int(row[1][0].split(" ")[1])
    x2 = int(row[1][0].split(" ")[2])
    y2 = int(row[1][0].split(" ")[3])
    cv2.rectangle(img,(x1,y1),(x2,y2),(255,0,0), 2)
#plt.figure()
#plt.imshow(img)"""

In [ ]:
"""import json


with open('/content/faster RCNN/test/airplane_228.json') as f:
  veri = json.load(f)
print(len(veri["shapes"]))
print("\n")
print(veri["shapes"])
print("\n",type(veri["shapes"]))"""

In [ ]:
classes = ["airplane"]
dataset_dicts = []
filename =  "airplane_217"                       #for filename in [file for file in os.listdir(directory) if file.endswith('.json')]:
json_file = "/content/faster RCNN/test/airplane_217.json"
with open(json_file) as f:
    img_anns = json.load(f)
record = {}


record["file_name"] = filename
record["height"] = 256
record["width"] = 256
annos = img_anns["shapes"] #shapes
objs = []
for anno in annos:
    px = [a[0] for a in anno['points']] # x coord
    py = [a[1] for a in anno['points']] # y-coord

    obj = {
        "bbox": [np.min(px), np.min(py), np.max(px), np.max(py)],
        "bbox_mode": BoxMode.XYXY_ABS,
        "category_id": classes.index(anno['label']),
        "iscrowd": 0
          }
    objs.append(obj)
record["annotations"] = objs
dataset_dicts.append(record)

In [ ]:
dataset_dicts

In [ ]:
os.mkdir("predict annot4")

In [ ]:
from matplotlib.pyplot import figure
namelist = {}


img = cv2.imread("/content/drive/MyDrive/Colab_Notebooks/train_data3/10238/Seri4/10238-Seri4-39830356056.jpeg")

outputs = predictor(img)
v = Visualizer(img[:, :, ::-1],
               metadata=microcontroller_metadata,
               scale=0.8,
              # instance_mode=ColorMode.IMAGE_BW # removes the colors of unsegmented pixels
)
v = v.draw_instance_predictions(outputs["instances"].to("cpu"))
plt.figure(figsize = (10, 10))
plt.axis("off")
plt.imshow(cv2.cvtColor(v.get_image()[:, :, ::-1], cv2.COLOR_BGR2RGB))
#plt.title(d["file_name"].split("/")[4])       #file name ayırma
plt.show()


## for x in range(len(outputs["instances"])):
#    #print(x)
#    deneme = outputs["instances"][x]
#    print(deneme)

#             orjinal konumları yazdırma
#  print(d)
#  annot = str(d)
#  print(annot,"\n")
#  print(annot.split("annotations")[1])



#             tahmin değerlerini txt dosyasına  yazdırma
b = str(outputs["instances"])
predictannot = b.split("tensor")[1].strip(" device='cuda:0')), scores:")
print(b,"\n")# dosya formatını görmek isterseniz açın
print(predictannot)
#filename = str(d["file_name"].split("/")[4].strip(".jpeg"))
#print(filename)
"""
dosya = open("/content/predict annot4/"+filename +".json", "w")
dosya.write(predictannot)
dosya.close()"""